# G2a · Candidate pool and full test streams for Seam 2 (checklist 2.0)

G1b showed that a fitted text-to-image map adds nothing, and that the gain lies in near-OOD negative labels
that are **grounded in the test stream**: each label's prototype becomes the centroid of the test images it
picks up. To build and tune that (Seam 2) on the real features, the local analysis needs two things this notebook
exports to `MyDrive/ReNeg/transfers/analysis_pack2/`:

1. `reneg_pool.npz` — CLIP text embeddings of the 14,526 WordNet near-OOD candidates (siblings and cousins of
   the ImageNet classes; the classes themselves, their ancestors and descendants removed), plus the words
   behind NegLabel's 10,000 negatives.
2. `reneg_stream_*.npz` — every image of ImageNet test, SSB-hard, NINCO, Textures, iNaturalist and
   OpenImage-O, stored as 8-bit codes (half the size of the cache, accurate to about 0.0003 in cosine).
3. `baseline_results.zip` — the CSVs from your Kaggle notebooks 02–04 (TANL and the other baselines), so
   Seam 2 is compared against your own runs, plus the oodlab code (ReNeg is built on its TANL).

**Runtime:** CPU or GPU, about 5 minutes. Then attach every file the last cell lists (each under 25 MB).

In [ ]:
# ── Colab set-up (the same cell in every ReNeg notebook) ─────────────────────────────
import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
HOME = os.environ.get("RENEG_OODLAB_HOME", "/content/oodlab_home" if IN_COLAB else os.path.abspath("oodlab_home"))
for d in (f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/scratch", f"{HOME}/input"):
    os.makedirs(d, exist_ok=True)
if not os.path.islink(f"{HOME}/working"):              # working/ lives on Drive, scratch/ stays local
    if os.path.isdir(f"{HOME}/working") and not os.listdir(f"{HOME}/working"):
        os.rmdir(f"{HOME}/working")
    if not os.path.exists(f"{HOME}/working"):
        os.symlink(f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/working")
os.environ["OODLAB_HOME"] = HOME
# a newer ReNeg_Colab_bundle*.zip in uploads/ is installed here, so a new bundle needs no C0 re-run
import glob, re, shutil, zipfile
def _reneg_version(text):
    m = re.search(r'__version__ = "([0-9.]+)"', text or "")
    return tuple(int(x) for x in m.group(1).split(".")) if m else None
_init = f"{DRIVE_ROOT}/code/reneg_code/reneg/__init__.py"
_installed = _reneg_version(open(_init).read()) if os.path.isfile(_init) else None
_newest = None
for _zp in glob.glob(f"{DRIVE_ROOT}/uploads/*.zip"):
    try:
        with zipfile.ZipFile(_zp) as _z:
            _hit = [n for n in _z.namelist() if n.endswith("reneg_code/reneg/__init__.py")]
            _v = _reneg_version(_z.read(_hit[0]).decode()) if _hit else None
        if _v and (_newest is None or _v > _newest[0]):
            _newest = (_v, _zp, _hit[0][: -len("reneg_code/reneg/__init__.py")])
    except (zipfile.BadZipFile, OSError):
        pass
if _newest and (_installed is None or _newest[0] > _installed):
    _tmp = os.path.join(HOME, "scratch", "_bundle")
    shutil.rmtree(_tmp, ignore_errors=True)
    with zipfile.ZipFile(_newest[1]) as _z:
        _z.extractall(_tmp)
    _src = os.path.join(_tmp, _newest[2])
    shutil.rmtree(f"{DRIVE_ROOT}/code/reneg_code", ignore_errors=True)
    shutil.copytree(os.path.join(_src, "reneg_code"), f"{DRIVE_ROOT}/code/reneg_code")
    os.makedirs(f"{DRIVE_ROOT}/notebooks", exist_ok=True)
    for _f in os.listdir(os.path.join(_src, "notebooks")):
        if _f.endswith(".ipynb") and not _f.startswith("C0"):
            shutil.copy2(os.path.join(_src, "notebooks", _f), f"{DRIVE_ROOT}/notebooks/{_f}")
    _fmt = lambda v: ".".join(map(str, v)) if v else "none"
    print(f"reneg code updated {_fmt(_installed)} -> {_fmt(_newest[0])} from uploads/{os.path.basename(_newest[1])}")
    if "reneg" in sys.modules:
        print("reneg was already imported in this session: Runtime -> Restart session, then run again")
CODE_ROOT = f"{DRIVE_ROOT}/code/oodlab_code"           # same name as in the Kaggle notebooks
for p in (CODE_ROOT, f"{DRIVE_ROOT}/code/reneg_code"):
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} not found: run C0_transfer_from_kaggle first (GUIDE_COLAB.md, step 4)")
    if p not in sys.path:
        sys.path.insert(0, p)
import oodlab, reneg
print(f"oodlab {oodlab.__version__} | reneg {reneg.__version__} | project folder {DRIVE_ROOT}")

In [ ]:
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True)
ctx = start("G2a_pool_and_streams", need_gpu=False)
log = ctx.log
import glob, json, zipfile
from oodlab.report import find_cache
from oodlab.clipwrap import load_clip
from reneg.g1 import clip_text_encoder
from reneg.packs import export_pool, export_streams, load_kg_pool
BACKBONE = "ViT-B/16"
OUT = f"{DRIVE_ROOT}/transfers/analysis_pack2"
os.makedirs(OUT, exist_ok=True)
cache = find_cache(ctx, BACKBONE)
bank = cache.load_textbank()
pool = load_kg_pool()
print(len(pool["pool"]), "candidate names; e.g.", [r["name"] for r in pool["pool"][:5]])

## 1 · Encode the candidate pool (same prompt as the text bank)

In [ ]:
WEIGHTS = f"{DRIVE_ROOT}/weights"
model = load_clip(BACKBONE, device=ctx.device, input_roots=ctx.input_roots + [WEIGHTS], download_root=WEIGHTS)
enc = clip_text_encoder(model, prompt="The nice {}.")
import torch
check = float((enc(list(bank.id_names[:20])) * torch.nn.functional.normalize(bank.id_text[:20].float(), dim=1)).sum(1).min())
print(f"encoder reproduces the bank's ID embeddings: min cosine {check:.6f}")
pool_path = export_pool(bank, enc, OUT, pool=pool, prompt_note="The nice {}.", log=log.info)

## 2 · Export the full test streams (8-bit)

In [ ]:
SETS = ["imagenet_test", "ssb_hard", "ninco", "textures", "inaturalist", "openimage_o"]
stream_paths = export_streams(cache, OUT, SETS, max_mb=24.0, log=log.info)

## 3 · Your baseline results (Kaggle notebooks 02–04)

In [ ]:
csvs = sorted(p for p in glob.glob(f"{ctx.dirs.results}/0[2-4]*/**/*.csv", recursive=True))
zpath = os.path.join(OUT, "baseline_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in csvs:
        z.write(p, os.path.relpath(p, ctx.dirs.results))
    # the oodlab code too (TANL, AdaNeg, NegLabel): ReNeg is built as an extension of this TANL
    for dp, dn, fn in os.walk(CODE_ROOT):
        dn[:] = [d for d in dn if d not in ("__pycache__", ".git", ".pytest_cache")]
        for f in fn:
            if not f.endswith((".pyc", ".npz", ".pt")):
                q = os.path.join(dp, f)
                z.write(q, os.path.join("oodlab_code", os.path.relpath(q, CODE_ROOT)))
print(f"{len(csvs)} CSV files and the oodlab code -> {zpath} ({os.path.getsize(zpath) / 1e6:.1f} MB)")
for p in csvs:
    print("   ", os.path.relpath(p, ctx.dirs.results))

## Files written

In [ ]:
files = [pool_path] + stream_paths + [zpath]
total = 0
for p in files:
    mb = os.path.getsize(p) / 1e6
    total += mb
    print(f"{mb:6.1f} MB  {p}")
print(f"{len(files)} files, {total:.0f} MB in total, all in {OUT}")
finish(ctx, {"files": [os.path.basename(p) for p in files]})

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")